In [1]:
# -*- coding: utf-8 -*-
"""
Create LaTeX table from Hyb-Adam-UM hyperparameter-sensitivity output.

Input:
    hyb_adam_um_outputs/hyperparameter_sensitivity/
        hyb_adam_um_hyperparam_sensitivity_summary.csv

Fallback input:
    hyb_adam_um_outputs/hyperparameter_sensitivity/
        hyb_adam_um_hyperparam_sensitivity_detailed.csv

Output:
    hyb_adam_um_outputs/hyperparameter_sensitivity/
        hyb_adam_um_hyperparam_sensitivity_table.tex

Also saves:
    hyb_adam_um_outputs/hyperparameter_sensitivity/
        hyb_adam_um_hyperparam_sensitivity_table_check.csv

LaTeX packages:
    \\usepackage{booktabs}
    \\usepackage{graphicx}
"""

from __future__ import annotations

import os
from typing import Dict, List, Tuple

import numpy as np
import pandas as pd


# ============================================================
# -------------------------- CONFIG ---------------------------
# ============================================================

SENSITIVITY_DIR = os.path.join(
    "hyb_adam_um_outputs",
    "hyperparameter_sensitivity",
)

SUMMARY_CSV = os.path.join(
    SENSITIVITY_DIR,
    "hyb_adam_um_hyperparam_sensitivity_summary.csv",
)

DETAILED_CSV = os.path.join(
    SENSITIVITY_DIR,
    "hyb_adam_um_hyperparam_sensitivity_detailed.csv",
)

OUT_TEX = os.path.join(
    SENSITIVITY_DIR,
    "hyb_adam_um_hyperparam_sensitivity_table.tex",
)

OUT_CHECK_CSV = os.path.join(
    SENSITIVITY_DIR,
    "hyb_adam_um_hyperparam_sensitivity_table_check.csv",
)

# Include old initialization / restart ablations in the table.
# For the reviewer response, False is usually cleaner.
INCLUDE_INIT_ABLATIONS = False

# If your main paper reports RMSE/MAE as ×10^{-2}, keep True.
SCALE_ERRORS_X100 = True

# Bold best values in each metric column.
BOLD_BEST = True

# Use resizebox so the table fits the page.
USE_RESIZEBOX = True

TABLE_LABEL = "tab:hyb_adam_sensitivity"

if SCALE_ERRORS_X100:
    TABLE_CAPTION = (
        "Hyperparameter sensitivity of Hyb-Adam-UM at 50\\% missingness. "
        "All variants are evaluated on the same frozen masks. "
        "Matrix errors are computed only on artificially hidden entries. "
        "RMSE and MAE are reported as $\\times 10^{-2}$. "
        "Values are reported as mean $\\pm$ standard deviation."
    )
else:
    TABLE_CAPTION = (
        "Hyperparameter sensitivity of Hyb-Adam-UM at 50\\% missingness. "
        "All variants are evaluated on the same frozen masks. "
        "Matrix errors are computed only on artificially hidden entries. "
        "Values are reported as mean $\\pm$ standard deviation."
    )


# ============================================================
# ---------------------- TABLE VARIANTS -----------------------
# ============================================================

VARIANT_ORDER: List[str] = [
    "baseline",
    "lr_init_low",
    "lr_init_high",
    "schedule_fast_decay",
    "schedule_slow_decay",
    "clip_low",
    "clip_high",
    "clip_none",
    "fd_step_small",
    "fd_step_large",
]

INIT_ABLATION_ORDER: List[str] = [
    "old_global_mean_init",
    "single_restart",
]

VARIANT_DISPLAY: Dict[str, str] = {
    "baseline": "Baseline",
    "old_global_mean_init": "Global-mean init.",
    "single_restart": "Single restart",
    "lr_init_low": "Low learning rate",
    "lr_init_high": "High learning rate",
    "schedule_fast_decay": "Fast LR decay",
    "schedule_slow_decay": "Slow LR decay",
    "clip_low": "Clip = 1.0",
    "clip_high": "Clip = 10.0",
    "clip_none": "No clipping",
    "fd_step_small": "FD step $10^{-5}$",
    "fd_step_large": "FD step $10^{-4}$",
}


# ============================================================
# -------------------------- HELPERS --------------------------
# ============================================================

def ensure_parent(path: str) -> None:
    parent = os.path.dirname(path)
    if parent:
        os.makedirs(parent, exist_ok=True)


def finite_float(x) -> float:
    try:
        val = float(x)
    except Exception:
        return float("nan")
    return val if np.isfinite(val) else float("nan")


def mean_std_nan(values: pd.Series) -> Tuple[float, float]:
    arr = pd.to_numeric(values, errors="coerce").to_numpy(dtype=float)
    arr = arr[np.isfinite(arr)]

    if len(arr) == 0:
        return float("nan"), float("nan")

    mean_val = float(np.mean(arr))
    std_val = float(np.std(arr, ddof=1)) if len(arr) > 1 else 0.0

    return mean_val, std_val


def latex_escape_text(s: str) -> str:
    """
    Escape ordinary text for LaTeX.
    Math strings containing $...$ are preserved.
    """
    s = str(s)

    if "$" in s:
        return s

    replacements = {
        "\\": r"\textbackslash{}",
        "&": r"\&",
        "%": r"\%",
        "#": r"\#",
        "_": r"\_",
        "{": r"\{",
        "}": r"\}",
        "~": r"\textasciitilde{}",
        "^": r"\textasciicircum{}",
    }

    return "".join(replacements.get(ch, ch) for ch in s)


def fmt_pm(
    mean_val: float,
    std_val: float,
    decimals: int,
    scale: float = 1.0,
    bold: bool = False,
) -> str:
    """
    Format mean ± std for LaTeX.
    """
    mean_val = finite_float(mean_val)
    std_val = finite_float(std_val)

    if not np.isfinite(mean_val):
        return "--"

    mean_val *= scale

    if np.isfinite(std_val):
        std_val *= scale
        core = f"{mean_val:.{decimals}f} $\\pm$ {std_val:.{decimals}f}"
    else:
        core = f"{mean_val:.{decimals}f}"

    if bold:
        return r"\textbf{" + core + "}"

    return core


def safe_int(x, default: int = 0) -> int:
    try:
        if pd.isna(x):
            return default
        return int(x)
    except Exception:
        return default


# ============================================================
# -------------------- LOAD / BUILD SUMMARY -------------------
# ============================================================

def load_or_build_summary(summary_csv: str, detailed_csv: str) -> pd.DataFrame:
    """
    Prefer the summary CSV generated by the sensitivity code.
    If absent, rebuild a compatible summary from the detailed CSV.
    """
    if os.path.exists(summary_csv):
        print(f"Reading summary file:\n  {summary_csv}")
        return pd.read_csv(summary_csv)

    if not os.path.exists(detailed_csv):
        raise FileNotFoundError(
            "Neither summary nor detailed sensitivity file was found.\n"
            f"Tried:\n  {summary_csv}\n  {detailed_csv}"
        )

    print("Summary file not found. Rebuilding summary from detailed file:")
    print(f"  {detailed_csv}")

    detailed = pd.read_csv(detailed_csv)

    if "sensitivity_variant" not in detailed.columns:
        raise ValueError("Detailed file does not contain 'sensitivity_variant'.")

    rows = []

    for variant, group0 in detailed.groupby("sensitivity_variant", sort=False):
        if "success" in group0.columns:
            success_mask = group0["success"].astype(bool)
        else:
            success_mask = pd.Series(True, index=group0.index)

        group = group0[success_mask].copy()

        row = {
            "sensitivity_variant": variant,
            "variant_description": (
                group0["variant_description"].iloc[0]
                if "variant_description" in group0.columns
                else ""
            ),
            "n_runs": int(len(group0)),
            "n_success": int(success_mask.sum()),
            "n_failed": int(len(group0) - success_mask.sum()),
        }

        for col in [
            "RMSE_miss",
            "MAE_miss",
            "Pearson_miss",
            "Spearman_miss",
            "Delta_final",
            "runtime_seconds",
        ]:
            if col in group.columns:
                mean_val, std_val = mean_std_nan(group[col])
            else:
                mean_val, std_val = float("nan"), float("nan")

            row[f"{col}_mean"] = mean_val
            row[f"{col}_std"] = std_val

        rows.append(row)

    return pd.DataFrame(rows)


# ============================================================
# ---------------------- BEST VALUE MASKS ---------------------
# ============================================================

def get_best_masks(df: pd.DataFrame) -> Dict[str, pd.Series]:
    """
    Return boolean Series identifying best values.

    Lower is better:
        RMSE, MAE, final Delta, runtime.

    Higher is better:
        Pearson, Spearman.
    """
    best_masks: Dict[str, pd.Series] = {}

    directions = {
        "RMSE_miss_mean": "min",
        "MAE_miss_mean": "min",
        "Pearson_miss_mean": "max",
        "Spearman_miss_mean": "max",
        "Delta_final_mean": "min",
        "runtime_seconds_mean": "min",
    }

    for col, direction in directions.items():
        if col not in df.columns:
            best_masks[col] = pd.Series(False, index=df.index)
            continue

        vals = pd.to_numeric(df[col], errors="coerce")
        finite_vals = vals[np.isfinite(vals)]

        if finite_vals.empty:
            best_masks[col] = pd.Series(False, index=df.index)
            continue

        if direction == "min":
            best_val = finite_vals.min()
        else:
            best_val = finite_vals.max()

        mask_array = np.isclose(
            vals.to_numpy(dtype=float),
            float(best_val),
            rtol=1.0e-10,
            atol=1.0e-12,
        )

        mask = pd.Series(mask_array, index=df.index)
        mask = mask & vals.notna()

        best_masks[col] = mask

    return best_masks


def is_best(best_masks: Dict[str, pd.Series], col: str, idx) -> bool:
    """
    Safely check whether row idx is best in column col.
    """
    if not BOLD_BEST:
        return False

    mask = best_masks.get(col)

    if mask is None:
        return False

    try:
        return bool(mask.loc[idx])
    except Exception:
        return False


# ============================================================
# ---------------------- PREPARE TABLE DF ---------------------
# ============================================================

def prepare_table_df(summary_df: pd.DataFrame) -> pd.DataFrame:
    df = summary_df.copy()

    if "sensitivity_variant" not in df.columns:
        raise ValueError("Summary file must contain 'sensitivity_variant'.")

    wanted_order = VARIANT_ORDER.copy()

    if INCLUDE_INIT_ABLATIONS:
        wanted_order = (
            ["baseline"]
            + INIT_ABLATION_ORDER
            + [v for v in VARIANT_ORDER if v != "baseline"]
        )

    known = df[df["sensitivity_variant"].isin(wanted_order)].copy()
    unknown = df[~df["sensitivity_variant"].isin(wanted_order)].copy()

    order_map = {v: i for i, v in enumerate(wanted_order)}

    known["__order__"] = known["sensitivity_variant"].map(order_map)
    known = known.sort_values("__order__")

    unknown["__order__"] = range(
        len(wanted_order),
        len(wanted_order) + len(unknown),
    )

    df = pd.concat([known, unknown], ignore_index=True)

    df["Variant"] = df["sensitivity_variant"].map(VARIANT_DISPLAY)
    df["Variant"] = df["Variant"].fillna(df["sensitivity_variant"])

    if "__order__" in df.columns:
        df = df.drop(columns=["__order__"])

    return df


# ============================================================
# ---------------------- LATEX GENERATION ---------------------
# ============================================================

def build_latex_table(df: pd.DataFrame) -> str:
    best_masks = get_best_masks(df)

    error_scale = 100.0 if SCALE_ERRORS_X100 else 1.0
    error_decimals = 2 if SCALE_ERRORS_X100 else 4

    lines: List[str] = []

    lines.append(r"\begin{table}[t]")
    lines.append(r"\centering")
    lines.append(r"\caption{" + TABLE_CAPTION + "}")
    lines.append(r"\label{" + TABLE_LABEL + "}")
    lines.append(r"\small")

    if USE_RESIZEBOX:
        lines.append(r"\resizebox{\textwidth}{!}{%")

    lines.append(r"\begin{tabular}{lccccccc}")
    lines.append(r"\toprule")
    lines.append(
        r"Variant & "
        r"RMSE$_{\rm miss}$ & "
        r"MAE$_{\rm miss}$ & "
        r"Pearson$_{\rm miss}$ & "
        r"Spearman$_{\rm miss}$ & "
        r"Final $\Delta$ & "
        r"Time (s) & "
        r"Success \\"
    )
    lines.append(r"\midrule")

    for idx, row in df.iterrows():
        variant = latex_escape_text(row["Variant"])

        rmse = fmt_pm(
            row.get("RMSE_miss_mean", np.nan),
            row.get("RMSE_miss_std", np.nan),
            decimals=error_decimals,
            scale=error_scale,
            bold=is_best(best_masks, "RMSE_miss_mean", idx),
        )

        mae = fmt_pm(
            row.get("MAE_miss_mean", np.nan),
            row.get("MAE_miss_std", np.nan),
            decimals=error_decimals,
            scale=error_scale,
            bold=is_best(best_masks, "MAE_miss_mean", idx),
        )

        pearson = fmt_pm(
            row.get("Pearson_miss_mean", np.nan),
            row.get("Pearson_miss_std", np.nan),
            decimals=3,
            scale=1.0,
            bold=is_best(best_masks, "Pearson_miss_mean", idx),
        )

        spearman = fmt_pm(
            row.get("Spearman_miss_mean", np.nan),
            row.get("Spearman_miss_std", np.nan),
            decimals=3,
            scale=1.0,
            bold=is_best(best_masks, "Spearman_miss_mean", idx),
        )

        delta = fmt_pm(
            row.get("Delta_final_mean", np.nan),
            row.get("Delta_final_std", np.nan),
            decimals=2,
            scale=1.0,
            bold=is_best(best_masks, "Delta_final_mean", idx),
        )

        runtime = fmt_pm(
            row.get("runtime_seconds_mean", np.nan),
            row.get("runtime_seconds_std", np.nan),
            decimals=2,
            scale=1.0,
            bold=is_best(best_masks, "runtime_seconds_mean", idx),
        )

        n_success = safe_int(row.get("n_success", 0), default=0)
        n_runs = safe_int(row.get("n_runs", n_success), default=n_success)
        success = f"{n_success}/{n_runs}"

        lines.append(
            f"{variant} & "
            f"{rmse} & "
            f"{mae} & "
            f"{pearson} & "
            f"{spearman} & "
            f"{delta} & "
            f"{runtime} & "
            f"{success} \\\\"
        )

    lines.append(r"\bottomrule")
    lines.append(r"\end{tabular}")

    if USE_RESIZEBOX:
        lines.append(r"}")

    lines.append(r"\end{table}")

    return "\n".join(lines) + "\n"


# ============================================================
# -------------------------- MAIN -----------------------------
# ============================================================

def main() -> None:
    summary_df = load_or_build_summary(SUMMARY_CSV, DETAILED_CSV)
    table_df = prepare_table_df(summary_df)

    check_cols = [
        "sensitivity_variant",
        "Variant",
        "variant_description",
        "n_runs",
        "n_success",
        "n_failed",
        "RMSE_miss_mean",
        "RMSE_miss_std",
        "MAE_miss_mean",
        "MAE_miss_std",
        "Pearson_miss_mean",
        "Pearson_miss_std",
        "Spearman_miss_mean",
        "Spearman_miss_std",
        "Delta_final_mean",
        "Delta_final_std",
        "runtime_seconds_mean",
        "runtime_seconds_std",
    ]

    existing_check_cols = [c for c in check_cols if c in table_df.columns]

    ensure_parent(OUT_CHECK_CSV)
    table_df[existing_check_cols].to_csv(OUT_CHECK_CSV, index=False)

    latex = build_latex_table(table_df)

    ensure_parent(OUT_TEX)
    with open(OUT_TEX, "w", encoding="utf-8") as f:
        f.write(latex)

    print("\nSaved LaTeX table:")
    print(f"  {OUT_TEX}")

    print("\nSaved check CSV:")
    print(f"  {OUT_CHECK_CSV}")

    print("\nLaTeX preview:\n")
    print(latex)


if __name__ == "__main__":
    main()

Reading summary file:
  hyb_adam_um_outputs/hyperparameter_sensitivity/hyb_adam_um_hyperparam_sensitivity_summary.csv

Saved LaTeX table:
  hyb_adam_um_outputs/hyperparameter_sensitivity/hyb_adam_um_hyperparam_sensitivity_table.tex

Saved check CSV:
  hyb_adam_um_outputs/hyperparameter_sensitivity/hyb_adam_um_hyperparam_sensitivity_table_check.csv

LaTeX preview:

\begin{table}[t]
\centering
\caption{Hyperparameter sensitivity of Hyb-Adam-UM at 50\% missingness. All variants are evaluated on the same frozen masks. Matrix errors are computed only on artificially hidden entries. RMSE and MAE are reported as $\times 10^{-2}$. Values are reported as mean $\pm$ standard deviation.}
\label{tab:hyb_adam_sensitivity}
\small
\resizebox{\textwidth}{!}{%
\begin{tabular}{lccccccc}
\toprule
Variant & RMSE$_{\rm miss}$ & MAE$_{\rm miss}$ & Pearson$_{\rm miss}$ & Spearman$_{\rm miss}$ & Final $\Delta$ & Time (s) & Success \\
\midrule
Baseline & 1.34 $\pm$ 0.73 & 0.51 $\pm$ 0.21 & 0.925 $\pm$ 0.064 & 